# Learning in a Turn-Based Game

In the previous notebook the agent still lost some games against a random opponent. A natural explanation
is that a random opponent is a weak teacher. Before accepting it, we check something more basic: **is the
training loop feeding the update rule the right information?**

We will instrument the agent to see which updates it actually receives, find two problems, fix them, and
measure the difference.

## Setup

The reusable code lives in the `core` package, one folder up (`04_qlearning/core/`). We add that folder
to Python's import path so that `from core... import ...` works from this notebook. A fixed random seed
makes the results reproducible.

In [1]:
import sys
import random
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
random.seed(42)

This is the training loop from notebook 01, unchanged: the agent is updated right after each of its own moves.

In [2]:
from core.environment import TicTacToeEnvironment
from core.agents.q_learning_agent import QLearningAgent

In [3]:
def play_training_game(agent: QLearningAgent, env: TicTacToeEnvironment, agent_first: bool = True) -> str:
    """Play one game against a random opponent, updating the agent after each of its moves."""
    env.reset()
    agent.reset_episode()

    agent_symbol = agent.symbol
    opponent_symbol = 'O' if agent_symbol == 'X' else 'X'
    current_player = agent_symbol if agent_first else opponent_symbol

    while True:
        available = env.get_available_actions()
        if current_player == agent_symbol:
            move = agent.choose_action(env.board, available)
        else:
            move = random.choice(available)

        env.make_move(move, current_player)
        is_over, result = env.is_game_over()

        if current_player == agent_symbol:
            if is_over:
                # The agent's move ended the game: learn from the final reward
                reward = 1.0 if result == agent_symbol else 0.0
                agent.learn(reward=reward, next_board=None)
                return result
            # The game goes on: learn from the step penalty and the board the move produced
            agent.learn(reward=-0.01, next_board=env.board)
        elif is_over:
            return result

        current_player = opponent_symbol if current_player == agent_symbol else agent_symbol

## Instrumenting the Agent

`InstrumentedAgent` is a `QLearningAgent` that counts its updates before applying them:

- **Terminal updates**, by reward: +1 (win), 0 (draw), −1 (loss).
- **Non-terminal updates** (bootstraps), depending on whether $\max_{a'} Q(s', a')$ read a value the agent had
  already learned or just the initial 0.

In [4]:
from collections import Counter


class InstrumentedAgent(QLearningAgent):
    """A QLearningAgent that counts what kind of updates it receives."""

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.update_counts: Counter[str] = Counter()

    def learn(self, reward: float, next_board: dict[int, str | None] | None = None) -> None:
        if self.training_mode and self.episode_history:
            if next_board is None:
                self.update_counts[f"terminal reward {reward:+.0f}"] += 1
            else:
                next_state = self.get_state_key(next_board)
                next_actions = [cell for cell in range(1, 10) if next_board[cell] is None]
                has_known_value = any(self.get_q_value(next_state, action) != 0 for action in next_actions)
                self.update_counts["bootstrap with a known value" if has_known_value else "bootstrap from zero"] += 1
        super().learn(reward, next_board)

In [5]:
def train(game_function, agent: QLearningAgent, n_episodes: int = 20000) -> QLearningAgent:
    """Train `agent` against a random opponent with the given game loop and the standard ε schedule."""
    env = TicTacToeEnvironment()
    epsilon = 1.0
    for _ in range(n_episodes):
        epsilon = max(0.05, epsilon * 0.9995)
        agent.set_epsilon(epsilon)
        game_function(agent, env, agent_first=random.choice([True, False]))
    return agent

In [6]:
naive_agent = train(play_training_game, InstrumentedAgent(symbol='X'))

for update_type, count in sorted(naive_agent.update_counts.items()):
    print(f"{update_type:30} {count:6d}")

bootstrap from zero             31691
bootstrap with a known value    24196
terminal reward +0                861
terminal reward +1              13378


## Problem 1: The Agent Never Receives a Loss

There is no `terminal reward -1` line. In 20,000 games the agent never received a single −1, although it
lost about one game in five (notebook 01).

Look at the loop: the agent only learns after **its own** moves. Its own move can win or draw the game,
but never lose it. A loss always comes from the **opponent's** move, and then the loop just returns. The −1
reward is defined, but it is never delivered.

As far as the Q-table is concerned, losing does not exist, so the agent cannot learn to avoid it.

## Problem 2: The Next State Belongs to the Opponent

In the naive loop, $s'$ is the board **right after the agent's move**, so it is the **opponent** who is to
move there. Yet the update asks for $\max_{a'} Q(s', a')$: the value of the agent's best move in a
position where the agent does not move.

Where do the 24,196 "known values" (43% of the bootstraps) come from? A Q-value is only non-zero if the
agent once chose an action in that exact board, which means the agent was the one to move there. This can
happen because the starter is random. The same board is "opponent to move" in a game the agent started,
and "agent to move" in a game the opponent started. The update then borrows the value of **having the
move** in that position, which ignores the opponent's reply and is too optimistic.

The other 57% of the bootstraps read the initial 0, so for them γ has no effect at all.

In both cases, $\max_{a'} Q(s', a')$ is **not** the value of the position the agent will actually face
next.

## The Fix: The Opponent Is Part of the Environment

From a single agent's point of view, the opponent is just part of the environment. When the agent makes a
move, the environment "responds" with the opponent's move, and only then does the agent observe its next
state. So one transition spans **the agent's move and the opponent's reply**:

```
 my turn          opponent's turn        my turn again
 s_t ──(my action a_t)──► ... ──(opponent's reply)──► s_{t+1}
 └──────────────── one transition: update Q(s_t, a_t) here ────┘
```

This gives two rules:

1. **Update at the start of the agent's next turn**, using the current board as $s'$. That board is a
   state where the agent is to move, so $\max_{a'} Q(s', a')$ is the value of its best move there.
2. **When the game ends, update with the final reward, whoever made the last move.** If the opponent's
   move wins, the agent's last action receives −1.

The agent class does not change. Only *when* the loop calls `learn()` changes.

In [7]:
from core.training import terminal_reward


def play_turn_based_game(agent: QLearningAgent, env: TicTacToeEnvironment, agent_first: bool = True) -> str:
    """Play one game against a random opponent, closing each transition at the agent's next turn."""
    env.reset()
    agent.reset_episode()

    agent_symbol = agent.symbol
    opponent_symbol = 'O' if agent_symbol == 'X' else 'X'
    current_player = agent_symbol if agent_first else opponent_symbol

    while True:
        available = env.get_available_actions()
        if current_player == agent_symbol:
            # The opponent has replied: the transition started by our previous move ends here
            agent.learn(reward=-0.01, next_board=env.board)
            move = agent.choose_action(env.board, available)
        else:
            move = random.choice(available)

        env.make_move(move, current_player)
        is_over, result = env.is_game_over()

        if is_over:
            # Whoever made the last move, the agent's last action led to this outcome
            agent.learn(reward=terminal_reward(result, agent_symbol), next_board=None)
            return result

        current_player = opponent_symbol if current_player == agent_symbol else agent_symbol

The first call to `learn()` in each game does nothing, because the episode history is still empty.

In [8]:
turn_based_agent = train(play_turn_based_game, InstrumentedAgent(symbol='X'))

for update_type, count in sorted(turn_based_agent.update_counts.items()):
    print(f"{update_type:30} {count:6d}")

bootstrap from zero              4278
bootstrap with a known value    43928
terminal reward +0               1746
terminal reward +1              15901
terminal reward -1               2353


- The agent now receives 2,353 terminal updates with −1: every loss reaches the last action that allowed it.
- 91% of the bootstraps (43,928 of 48,206) read a known value. That value was learned in the same kind of
  state being evaluated, one where the agent is to move.
- The rest read 0 simply because the position had not been visited yet.

## Comparing Both Loops

We now train fresh agents with both loops, for two values of γ, and evaluate them against two opponents:

- **Random**: measures how well the agent exploits mistakes.
- **Minimax**: a perfect player (the algorithm from `02_minimax`, notebook 03 uses it for training). Nobody
  can beat it, so the only question is how often the agent **loses**. A perfect agent draws every game.

To reduce the effect of luck, each configuration is trained with 3 different seeds and the results are
averaged. This takes a little while.

In [9]:
from statistics import mean

from core.agents.random_agent import RandomAgent
from core.agents.minimax_agent import MinimaxAgent
from core.training import evaluate_agent

N_SEEDS = 3


def compare(game_function, discount_factor: float) -> dict[str, float]:
    """Average evaluation of agents trained with `game_function` over several seeds."""
    evaluations = []
    for seed in range(N_SEEDS):
        random.seed(seed)
        agent = train(game_function, QLearningAgent(symbol='X', discount_factor=discount_factor))
        vs_random = evaluate_agent(agent, RandomAgent('O'), n_games=1000)
        vs_minimax = evaluate_agent(agent, MinimaxAgent('O'), n_games=300)
        evaluations.append((vs_random, vs_minimax))
    return {
        'win vs Random': mean(r['win_rate'] for r, _ in evaluations),
        'loss vs Random': mean(r['loss_rate'] for r, _ in evaluations),
        'loss vs Minimax': mean(m['loss_rate'] for _, m in evaluations),
    }

In [10]:
comparison = {
    (loop_name, gamma): compare(game_function, gamma)
    for loop_name, game_function in [('naive', play_training_game), ('turn-based', play_turn_based_game)]
    for gamma in [0.0, 0.9]
}

print(f"{'loop':12}{'γ':>5}{'win vs Random':>16}{'loss vs Random':>16}{'loss vs Minimax':>17}")
for (loop_name, gamma), metrics in comparison.items():
    print(f"{loop_name:12}{gamma:>5}{metrics['win vs Random']:>16.1%}{metrics['loss vs Random']:>16.1%}"
          f"{metrics['loss vs Minimax']:>17.1%}")

loop            γ   win vs Random  loss vs Random  loss vs Minimax
naive         0.0           66.3%           26.3%            87.3%
naive         0.9           74.2%           21.2%            86.7%
turn-based    0.0           74.9%           12.9%            68.4%
turn-based    0.9           89.9%            2.3%            10.8%


- **The naive loop is poor with either γ.** It loses 21-26% against Random and about 87% against Minimax.
  γ changes little, as expected when most bootstraps read 0 or a value from the wrong turn.
- **The turn-based loop with γ=0.9** cuts the loss rate against Random from 21.2% to 2.3%, and against
  Minimax from 86.7% to 10.8%.
- **γ now matters.** With γ=0 the agent only learns from immediate rewards. It can learn to block a threat
  that loses on the next move, but not to avoid a move that leads to a lost position (for example a fork)
  a few turns later. It still loses 68.4% against Minimax. With γ=0.9, the final result propagates back
  to earlier moves.

The opponent was the same random player in all four rows, so most of the weakness of notebook 01 came from
the loop, not from the opponent. The remaining losses against Minimax are a **coverage** problem: a random
opponent rarely plays the strong lines that Minimax plays. That is the topic of the next notebooks.

## From Now On

`core.training.play_game` applies these two rules to a game between **any two agents**, so both can be
learners (self-play, notebooks 04 and 05):

In [11]:
import inspect

from core.training import play_game

print(inspect.getsource(play_game))

def play_game(agent_x: Agent, agent_o: Agent, env: TicTacToeEnvironment, x_starts: bool = True) -> str | None:
    """
    Play one game between two agents, letting every Q-learning agent learn from it.

    From a learner's point of view the opponent is part of the environment, so the transition
    started by its move only ends when it is its turn again (after the opponent's reply), or when
    the game ends. That is when each Q-learning update happens:

    - At the start of an agent's turn, its previous (state, action) is updated with the current
      board as next state, a board where that agent is to move.
    - When the game ends, both agents update their pending (state, action) with the final reward.

    Args:
        agent_x: Agent playing X
        agent_o: Agent playing O
        env: The game environment
        x_starts: If True, X plays first

    Returns:
        'X', 'O', or 'draw'
    """
    env.reset()
    learners = [agent for agent in (agent_x, agent_o) if isinst

Every notebook from here on trains with `play_game`.

**Takeaway:** in a multi-agent setting, the "environment" of each learner includes the other players. A
transition must go from one of *my* decision points to my *next* decision point. Otherwise the Bellman
update mixes up whose turn it is, and γ stops meaning what it should.